# 20 — Strategy Consultant

Give it a market name. A ReAct agent calls research tools, then synthesises a typed `MarketAnalysis`: TAM, competitors, scored opportunities/risks, and an ENTER / MONITOR / AVOID verdict.

In [ ]:
!pip install -q langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
# Schema
from typing import Literal
from pydantic import BaseModel, Field

class CompetitorProfile(BaseModel):
    name: str = Field(description='Competitor company name.')
    estimated_market_share_pct: float = Field(description='Estimated market share as a percentage (0-100).')
    strengths: list[str] = Field(description='Key competitive advantages.')
    weaknesses: list[str] = Field(description='Notable gaps or vulnerabilities.')

class OpportunityRisk(BaseModel):
    description: str = Field(description='Plain-English description.')
    score: int = Field(description='1-10 scale.', ge=1, le=10)
    category: Literal['opportunity', 'risk'] = Field(description='Opportunity or risk.')

class MarketAnalysis(BaseModel):
    market: str = Field(description='The market or geography.')
    market_size_usd_bn: float = Field(description='TAM in USD billions.')
    growth_rate_pct: float = Field(description='Annual growth rate %.')
    competitors: list[CompetitorProfile] = Field(description='Top 3-5 competitors.')
    opportunities_and_risks: list[OpportunityRisk] = Field(description='Scored items.')
    entry_recommendation: Literal['enter', 'monitor', 'avoid'] = Field(description='Verdict.')
    rationale: str = Field(description='Two-to-three sentence rationale.')

In [ ]:
# Tools + workflow
import json
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

@tool
def search_market_size(market: str) -> str:
    """Return estimated TAM and growth rate."""
    data = {
        'b2b saas europe': {'tam_usd_bn': 42.0, 'growth_pct': 18.5},
        'industrial iot usa': {'tam_usd_bn': 31.0, 'growth_pct': 22.3},
        'default': {'tam_usd_bn': 15.0, 'growth_pct': 12.0},
    }
    return json.dumps(data.get(market.lower(), data['default']))

@tool
def search_competitors(market: str) -> str:
    """Return top competitors."""
    c = {
        'b2b saas europe': [
            {'name': 'Salesforce', 'share_pct': 19.0, 'strengths': ['brand'], 'weaknesses': ['price']},
            {'name': 'HubSpot', 'share_pct': 11.0, 'strengths': ['UX'], 'weaknesses': ['enterprise gaps']},
        ],
    }
    return json.dumps(c.get(market.lower(), [{'name': 'Incumbent A', 'share_pct': 25.0, 'strengths': ['scale'], 'weaknesses': ['agility']}]))

@tool
def search_regulations(market: str) -> str:
    """Return regulatory overview."""
    r = {'b2b saas europe': 'GDPR + NIS2 apply.'}
    return r.get(market.lower(), 'Standard commercial regulations apply.')

_SYS = SystemMessage('You are a strategy consultant. Use tools to research the market, then summarise findings.')

def run(market):
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    agent = create_react_agent(llm, [search_market_size, search_competitors, search_regulations], prompt=_SYS)
    result = agent.invoke({'messages': [{'role': 'user', 'content': f'Analyse the market: {market}'}]})
    findings = result['messages'][-1].content
    return ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(MarketAnalysis).invoke(
        f'Convert these findings into a MarketAnalysis.\n\n{findings}\n\nMarket: {market}'
    )

In [ ]:
analysis = run('B2B SaaS Europe')
print(f'TAM: ${analysis.market_size_usd_bn:.1f}B | Growth: {analysis.growth_rate_pct:.1f}%')
print(f'Verdict: {analysis.entry_recommendation.upper()}')
print(f'Rationale: {analysis.rationale}')
print('\nCompetitors:')
for c in analysis.competitors:
    print(f'  {c.name}: {c.estimated_market_share_pct:.0f}% share')
print('\nOpportunities & Risks:')
for item in sorted(analysis.opportunities_and_risks, key=lambda x: -x.score):
    tag = '+' if item.category == 'opportunity' else '-'
    print(f'  [{tag}] {item.score}/10 {item.description}')

## Exercise

Swap in a real search tool (e.g. Exa or Tavily) for `search_market_size` and `search_competitors`. The schema and the synthesis step stay identical — only the data source changes.